# Neutral vs nonneutral: saved-feature follow-up

Run after classification_similarity.ipynb completes. **CPU, Run all.**

Neutral is one class; positive and negative are combined into nonneutral. Same word-alone/word-in-context agreement policy, all three holdouts separately, logistic regression, shrinkage LDA and a dummy baseline. No neutral downsampling. Balanced accuracy chance is 50%.

**No EEG reads or feature preparation.** Missing feature caches stop this notebook. The original positive/negative results are displayed for comparison. Additional outputs are expected to take a few to tens of MB, with a 60 MB check after each task (the first 25 MB limit was too low; completed cases resume). Saved under classification_similarity/neutral_nonneutral/.

**Resuming after the 25 MB error:** run all cells. This release recognizes the storage-only update and keeps completed model fits; no EEG reads or feature preparation. Expected total follow-up outputs are roughly 40–50 MB, with a 60 MB limit.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, json, html
import pandas as pd
from IPython.display import display, Markdown, Image, HTML

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
ARCHIVE=THESIS/'Results/word_sentiment_eeg/trial_archive_125hz'
OUT=THESIS/'Results/word_sentiment_eeg/classification_similarity'
state=json.loads((ARCHIVE/'manifest.json').read_text())
if state.get('status')!='completed' or state.get('completed_reader_recordings')!=48:
    raise RuntimeError('Preparation is not fully complete. Wait for all 48 recordings; no analysis has started.')
REPO=Path('/content/word-sentiment-eeg-neutral')
def git(*args):
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],check=True,
                   env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary analysis checkout has edits; preserve them before updating.')
    git('-C',REPO,'pull','--ff-only','origin','main')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-analysis.txt')],check=True)
print('Analysis code:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())
print('Archive complete:',state['completed_reader_recordings'],'recordings')


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/classify_neutral_nonneutral.py'),'--thesis-root',str(THESIS)]
run_with_progress(command,OUT/'neutral_nonneutral/runtime_status.json',activity='Neutral versus nonneutral classification using saved features')


## Binary results side by side

Scores are pooled out-of-fold balanced accuracies. Holdouts are separate experiments, not simultaneous exclusion of readers, words and sentences.


In [ ]:
FOLLOW=OUT/'neutral_nonneutral'
assert json.loads((FOLLOW/'manifest.json').read_text())['status']=='completed'
old=pd.read_csv(OUT/'scores.csv')
old=old[(old.policy=='word_agreement') & (old.target=='binary') & (old.feature=='post')].copy()
old['comparison']='positive vs negative'
new=pd.read_csv(FOLLOW/'scores.csv')
new=new[new.feature=='post'].copy()
new['comparison']='neutral vs nonneutral'
summary=pd.concat([old,new],ignore_index=True)
summary['balanced_accuracy_percent']=(summary.balanced_accuracy*100).round(2)
display(summary[['dataset','task','comparison','split','model','balanced_accuracy_percent','trials','class_counts']])
display(Markdown('### Pre-onset and word/fixation controls'))
controls=pd.read_csv(FOLLOW/'scores.csv')
display(controls[controls.feature.isin(['pre','nuisance'])][['dataset','task','feature','balanced_accuracy','roc_auc']])
skipped=json.loads((FOLLOW/'skipped.json').read_text())
if skipped: display(skipped)


## ZUCO SR: neutral/nonneutral confusion matrices


In [ ]:
plot=FOLLOW/'zuco'/'SR'/'confusions.png'
if plot.is_file(): display(Image(filename=str(plot)))


## ZUCO NR: neutral/nonneutral confusion matrices


In [ ]:
plot=FOLLOW/'zuco'/'NR'/'confusions.png'
if plot.is_file(): display(Image(filename=str(plot)))


## ZUCO TSR: neutral/nonneutral confusion matrices


In [ ]:
plot=FOLLOW/'zuco'/'TSR'/'confusions.png'
if plot.is_file(): display(Image(filename=str(plot)))


## TECO task1: neutral/nonneutral confusion matrices


In [ ]:
plot=FOLLOW/'teco'/'task1'/'confusions.png'
if plot.is_file(): display(Image(filename=str(plot)))
